# LELA60331 Week 7 Seminar, sheet 3 of 3: Training and evaluating a text classifier

In sheet 1 we trained logistic regression on simulated data, and in sheet 2 we turned 10,000 Yelp reviews into one-hot features. Now we put the two together: we train a classifier on the reviews and find out how well it works.

| Part | What you do |
|---|---|
| 0 | Run the set-up cells: everything you built in sheets 1 and 2 |
| 1 | Train a classifier on the reviews |
| 2 | Evaluate it on the test set |
| 3 | Inspect the weights, and improve the tokenisation |

As before, each activity has a cell to complete (replace every `???`) followed by a ✅ **check** cell.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used after each activity.
# You don't need to understand how it works: it prints ✅ when your answer is right and ❌ when it isn't yet.
def check(label, condition):
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 0 · Where we got to in sheets 1 and 2

The cells below set up everything you built in sheets 1 and 2, so that this sheet works on its own. Read through them to remind yourself of each piece, then run them in order.

| Name | What it is | Where |
|---|---|---|
| `classify(p)` | 1 if `p` is above 0.5, otherwise 0 | Week 5 |
| `average_loss(q, y)` | the average cross-entropy loss (it uses `loss_one`, which we redefine in Part 1) | Week 5 |
| `sparse_mat_vec(M, vector)` | a score for every row of a sparse matrix | sheet 2, Part 5 |
| `sparse_transpose_mat_vec(M, vector, num_columns)` | a gradient for every column of a sparse matrix | sheet 2, Part 5 |
| `tokenized_sents` | each review as a list of words | sheet 2, Part 2 |
| `type_list`, `word_to_index` | the 5000 vocabulary words, and the column of each | sheet 2, Part 2 |
| `train_ints`, `test_ints` | the indices of the 8000 training and 2000 test reviews | sheet 2, Part 3 |
| `y_train`, `y_test` | their labels, as 1 (positive) or 0 (negative) | sheet 2, Part 3 |
| `M_train`, `M_test` | their sparse one-hot matrices | sheet 2, Part 4 |

In [ ]:
import math
import random
import matplotlib.pyplot as plt

def classify(p):
    if p > 0.5:
        return 1
    else:
        return 0

def average_loss(q, y):
    losses = [loss_one(q[k], y[k]) for k in range(len(y))]
    return sum(losses) / len(losses)

def sparse_mat_vec(M, vector):
    result = []
    for row in M:
        total = 0
        for j in row:
            total += vector[j]
        result.append(total)
    return result

def sparse_transpose_mat_vec(M, vector, num_columns):
    result = [0] * num_columns
    for i in range(len(M)):
        for j in M[i]:
            result[j] += vector[i]
    return result

Now download and read the reviews, as in sheet 2:

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

In [ ]:
reviews = []
labels = []
with open("yelp_reviews.txt") as fp:
    lines = fp.readlines()
for line in lines[1:]:                         # skip the header line
    fields = line.rstrip().split("\t")         # [review, label]
    reviews.append(fields[0])
    labels.append(fields[1])
print(len(reviews), "reviews")

**Tokenisation.** This is the naive tokenisation from sheet 2: it just splits each review at spaces. You will come back to this cell in Part 3.

In [ ]:
tokenized_sents = []
for review in reviews:
    tokenized_sents.append(review.split())

Finally, the vocabulary, the training and test split and the one-hot matrices, exactly as in sheet 2. This takes a few seconds.

In [ ]:
# the 5000 most frequent words (sheet 2, Part 2)
counts = {}
for sent in tokenized_sents:
    for word in sent:
        counts[word] = counts.get(word, 0) + 1
pairs = sorted([(count, word) for word, count in counts.items()], reverse=True)
type_list = [word for count, word in pairs[:5000]]
word_to_index = {}
for j in range(len(type_list)):
    word_to_index[type_list[j]] = j

# 8000 training reviews and 2000 test reviews (sheet 2, Part 3)
random.seed(1)
train_ints = random.sample(range(len(reviews)), int(len(reviews) * 0.8))
train_set = set(train_ints)
test_ints = [i for i in range(len(reviews)) if i not in train_set]
y_train = [int(labels[i] == "positive") for i in train_ints]
y_test = [int(labels[i] == "positive") for i in test_ints]

# sparse one-hot matrices (sheet 2, Part 4)
def one_hot(token_lists):
    M = []
    for tokens in token_lists:
        row = set()
        for word in tokens:
            if word in word_to_index:
                row.add(word_to_index[word])
        M.append(sorted(row))
    return M

M_train = one_hot([tokenized_sents[i] for i in train_ints])
M_test = one_hot([tokenized_sents[i] for i in test_ints])
print(len(M_train), "training reviews,", sum(y_train), "of them positive")
print(len(M_test), "test reviews,", sum(y_test), "of them positive")

---
# Part 1 · Training a classifier on the reviews

We now have everything we need. The model works exactly like the two-feature model in sheet 1, but with 5000 weights, one for each word.

### Two small changes

**1. A safer sigmoid.** With 5000 features, `z` can be a very large positive or negative number. `math.exp(-z)` is then too big for Python to store when `z` is a large negative number. This version of `sigmoid` uses an equivalent formula for negative `z` that avoids the problem. It gives exactly the same values.

**2. A safer loss.** When `z` is large, the sigmoid can round to exactly 1 (or 0). If the right answer was given a probability of 0, `math.log2(0)` gives an error. So we add a tiny number, `eps`, inside the log.

Run the cell.

In [ ]:
def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    else:
        return math.exp(z) / (1 + math.exp(z))      # the same value, without calculating math.exp(-z)

eps = 0.00001          # a tiny number, so that we never take the log of 0

def loss_one(q, label):
    if label == 1:
        return -math.log2(q + eps)
    else:
        return -math.log2(1 - q + eps)

print(sigmoid(-1000), sigmoid(0), sigmoid(1000))      # 0.0 0.5 1.0, and no error
print(loss_one(1.0, 0))                               # a big loss, but no error

### Activity 1 — Predictions from an untrained model

As in Week 5, the model starts with random weights (and here a random bias too). Calculate, for every training review:

1. `z`: the score. *Hint:* `sparse_mat_vec` with `M_train` and `weights`, then add the bias to each number, as you did with `mat_vec` in sheet 1.
2. `q`: the probability, the sigmoid of each score
3. `predictions`: the class (1 or 0)
4. `accuracy`: the proportion of predictions that equal the labels `y_train`

Then print the loss with `average_loss`.

In [ ]:
random.seed(2)
num_features = len(type_list)
weights = [random.random() for j in range(num_features)]
bias = random.random()
num_samples = len(y_train)

z = ???
q = ???
predictions = ???
accuracy = ???
print(round(z[0], 3), round(q[0], 3), predictions[0])     # you should get 25.646 1.0 1
print("accuracy:", accuracy)                              # you should get 0.50325
print("loss:", round(average_loss(q, y_train), 3))        # you should get 8.071

In [ ]:
# ✅ Check
check("z", lambda: len(z) == 8000 and abs(z[0] - 25.646) < 0.001)
check("q", lambda: len(q) == 8000 and abs(q[0] - 1.0) < 0.001)
check("predictions", lambda: predictions[:5] == [classify(p) for p in q[:5]])
check("accuracy", lambda: accuracy == 0.50325)

Every weight starts positive, so every word pushes the score up. What does the model predict for every review? So why is the accuracy the number it is? (Look at the number of positive training reviews printed by the last set-up cell.)

### Activity 2 — One step of gradient descent

Now make one improvement, as in Week 5, Activities 8 and 9:

1. `errors`: `q - y_train` for every review
2. `dw`: the gradient for every weight. *Hint:* `sparse_transpose_mat_vec` with `M_train`, `errors` and `num_features`, then divide each number by `num_samples`.
3. `db`: the sum of the errors divided by the number of reviews
4. `new_weights` and `new_bias`: **new value = old value − learning rate × gradient**. *Hint:* for `new_weights`, a list comprehension over the positions `j` of the weights.
5. `new_q`: the probabilities with the new weights and bias

Has the loss gone down?

In [ ]:
lr = 0.4

errors = ???
dw = ???
db = ???

new_weights = ???
new_bias = ???

new_q = ???
print(round(dw[0], 3), round(db, 3))                                # you should get 0.488 0.496
print("loss before:", round(average_loss(q, y_train), 3))           # 8.071
print("loss after: ", round(average_loss(new_q, y_train), 3))       # you should get 8.007

In [ ]:
# ✅ Check
check("errors", lambda: len(errors) == 8000 and abs(errors[0] - (q[0] - y_train[0])) < 1e-9)
check("dw", lambda: len(dw) == 5000 and abs(dw[0] - 0.4884) < 0.001)
check("db", lambda: abs(db - 0.4957) < 0.001)
check("new_bias", lambda: abs(new_bias - (bias - 0.4 * db)) < 1e-9)
check("the loss went down", lambda: average_loss(new_q, y_train) < average_loss(q, y_train))

Word 0 is `type_list[0]`, the most frequent word. Why is its gradient so close to `db`? *Hint:* which reviews contain it?

### Activity 3 — The training loop

Now put the steps from Activities 1 and 2 into a loop, as you did for the two-feature model in sheet 1. Each pass through the loop:

1. calculates the probabilities `q` with the current weights and bias
2. records the loss
3. calculates the errors and the gradients
4. updates the weights and the bias

Training for 500 epochs takes about a minute in plain Python. While you wait, think about why each epoch takes so much longer than for the two-feature model in sheet 1.

In [ ]:
random.seed(2)
weights = [random.random() for j in range(num_features)]
bias = random.random()
lr = 0.4
n_epochs = 500
loss_history = []

for epoch in range(n_epochs):
    # 1. predictions
    q = ???
    # 2. loss
    loss_history.append(???)
    # 3. errors and gradients
    errors = ???
    dw = ???
    db = ???
    # 4. update
    weights = ???
    bias = ???
    if epoch % 50 == 0:
        print("epoch", epoch, " loss", round(loss_history[-1], 3))

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()
print("final loss:", round(loss_history[-1], 3))      # you should get 0.598

In [ ]:
# ✅ Check
check("500 epochs", lambda: len(loss_history) == 500)
check("the loss went down", lambda: loss_history[-1] < loss_history[0])
check("final loss about 0.598", lambda: abs(loss_history[-1] - 0.598) < 0.005)

The first epoch's loss is very high: the untrained model gives almost every negative review a probability of nearly 1 of being positive. Compare the shape of the curve with the one for the two-feature model in sheet 1.

---
# Part 2 · Evaluating on the test set

Now we see how well the model does on the 2000 test reviews, which it never saw during training.

### Activity 4 — Predictions for the test reviews

Calculate, for every test review, as in Activity 1 but with `M_test` and `y_test`:

1. `q_test`: the probability that it is positive
2. `y_test_pred`: the predicted class (1 or 0)
3. `accuracy_test`: the proportion of predictions that equal the labels

In [ ]:
q_test = ???
y_test_pred = ???
accuracy_test = ???
print("test accuracy:", accuracy_test)        # you should get 0.82

In [ ]:
# ✅ Check
check("q_test", lambda: len(q_test) == 2000 and all(0 <= p <= 1 for p in q_test))
check("y_test_pred", lambda: y_test_pred[:10] == [classify(p) for p in q_test[:10]])
check("accuracy_test", lambda: abs(accuracy_test - 0.82) < 0.001)

### Activity 5 — Precision and recall as functions

We will want precision and recall for the test set **and** for the training set, so this time we write them as **functions**, using the same counting as in sheet 1, Part 4.

1. Complete `confusion_counts(predicted, actual)`, which counts the true positives, false positives and false negatives. The condition for true positives is given.
2. Complete `precision(tp, fp)` and `recall(tp, fn)`.

A function can `return` several values, separated by commas. They come back as a tuple, which we can unpack into separate names: `tp, fp, fn = confusion_counts(...)` (Week 4 Python workbook).

In [ ]:
def confusion_counts(predicted, actual):
    tp = 0
    fp = 0
    fn = 0
    for k in range(len(predicted)):
        if predicted[k] == 1 and actual[k] == 1:
            tp += 1
        if ???:          # predicted positive, but really negative
            fp += 1
        if ???:          # predicted negative, but really positive
            fn += 1
    return tp, fp, fn

def precision(tp, fp):
    return ???

def recall(tp, fn):
    return ???

tp, fp, fn = confusion_counts(y_test_pred, y_test)
print(tp, fp, fn)                                        # you should get 807 193 167
print("test precision:", round(precision(tp, fp), 3))    # you should get 0.807
print("test recall:", round(recall(tp, fn), 3))          # you should get 0.829

In [ ]:
# ✅ Check
check("confusion_counts on a small example", lambda: confusion_counts([1, 1, 0, 0, 1], [1, 0, 1, 0, 1]) == (2, 1, 1))
check("precision", lambda: precision(2, 1) == 2 / 3)
check("recall", lambda: recall(2, 1) == 2 / 3)
check("the test counts add up", lambda: tp + fn == sum(y_test))

### Activity 6 — Performance on the training data

Now calculate precision and recall on the **training** data. You need the predicted classes for the training reviews first (as in Activity 4, but with `M_train`); then use your functions from Activity 5.

How do the numbers differ from those for the test set? Why do you think this is?

In [ ]:
q_train = ???
y_train_pred = ???
tp, fp, fn = ???
print("training precision:", round(precision(tp, fp), 3))     # you should get 0.857
print("training recall:", round(recall(tp, fn), 3))           # you should get 0.862

In [ ]:
# ✅ Check
check("training predictions", lambda: len(y_train_pred) == 8000)
check("training counts", lambda: (tp, fp, fn) == confusion_counts(y_train_pred, y_train))
check("better on the training data than on the test data",
      lambda: precision(tp, fp) > precision(*confusion_counts(y_test_pred, y_test)[:2]))

*Write your answer here.*

A model that does much better on its training data than on new data has **overfitted**: it has learned things that are true of the particular reviews it was trained on, but that don't hold in general.

---
# Part 3 · Inspecting the weights

With one-hot features, the weights are easy to interpret. Each weight belongs to one word: `weights[j]` is the weight for the word `type_list[j]`. Words with **high** weights are those whose presence makes the model most likely to say *positive*; words with **low** (very negative) weights are those whose presence makes it most likely to say *negative*.

To put the words in order of their weights, we make a list of `(weight, word)` pairs and sort it, just as we sorted `(count, word)` pairs in sheet 2. This time we **don't** reverse it, so it goes from the lowest weight to the highest. Run the cell.

In [ ]:
weight_word = [(weights[j], type_list[j]) for j in range(len(type_list))]
weight_word = sorted(weight_word)
print(weight_word[:3])         # the three lowest weights
print(weight_word[-3:])        # the three highest weights

### Activity 7 — The most negative and most positive words

1. `lowest`: a list of the **words** with the 20 lowest weights
2. `highest`: a list of the **words** with the 20 highest weights, starting with the highest. *Hint:* the slice `[::-1]` reverses a list.

In [ ]:
lowest = ???
highest = ???
print(lowest)
print(highest)

In [ ]:
# ✅ Check
check("20 lowest", lambda: len(lowest) == 20 and lowest[0] == weight_word[0][1])
check("20 highest, highest first", lambda: len(highest) == 20 and highest[0] == weight_word[-1][1])
check("words, not pairs", lambda: type(lowest[0]) == str and type(highest[0]) == str)

Do the lists make sense? Which words would you have expected to see?

### Activity 8 — Spotting overfitting

Look further down the lists: print the 100 lowest and 100 highest words. Do you spot any words that suggest overfitting: words that tell the model about these particular reviews, rather than about sentiment in general? *Hint:* look for numbers, names, and tokens that aren't really words.

In [ ]:
# your code here

*Write your observations here.*

### Activity 9 — Better tokenisation

Go back to the tokenisation cell in Part 0 (where `tokenized_sents` is built). Can you change the way the text is tokenised to get rid of the problematic tokens you noted in Activity 8? The string methods from the Week 2 Python workbook, such as `lower()`, `replace()`, `strip()` and `isalpha()`, will help.

Then run all the cells from there down to here again. Some ✅ checks will now show ❌, because the expected values were worked out with the original tokenisation: that's fine. How do your changes affect the test precision and recall, and the lists of words?

*Write your observations here.*

---
## ✏️ Summary

Across this week's three sheets you have finished building logistic regression, and used it on real text:

1. **training**: repeat the improvement step many times, with a sensible learning rate (sheets 1 and 3)
2. **evaluation**: accuracy, precision and recall, on data the model has never seen (sheets 1 and 3)
3. **matrix-vector multiplication**: a score for every text is the data matrix × the weights, and a gradient for every weight is the transpose × the errors, however many features there are (sheets 1 and 2)
4. **text as features**: tokenise, choose a vocabulary, and one-hot encode each text, storing only the columns that hold a 1, a **sparse** matrix (sheet 2)
5. **overfitting**: a model can do much better on its training data than on new data; inspecting the weights helps show why (sheet 3)